# PlantMetWiki — Data transformation pipeline overview

This notebook generates two publication-ready artefacts:

1. **Pipeline transformation table** — one row per pipeline stage, with tool name, output artefact, key statistics, and notes. Suitable as Table 1 or supplementary material.

2. **Pipeline figure** — a Graphviz diagram (SVG + PDF + DOT source) showing all stages from PlantCyc input to SPARQL access. The SVG is fully editable in Inkscape or Illustrator.

## How to run

```bash
conda activate plantmetwiki-rdf
jupyter notebook notebooks/pipeline_overview.ipynb
```

No running Virtuoso instance required — all numbers are embedded in `PIPELINE_STATS` (Cell 1) and updated once per release.

## Updating numbers after a new release

Edit the `PIPELINE_STATS` dictionary in **Cell 1** with new triple counts from the fresh RDF conversion and Virtuoso load.


In [1]:
import pandas as pd
import graphviz
from pathlib import Path

FIG_DIR = Path('figures/output/figures')
OUT_DIR = Path('figures/output')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Pipeline statistics — update this dict after each new RDF conversion ────
# Based on PlantCyc 17.0.0 → GPML 2021 (v3) → RDF → Virtuoso 7.2
PIPELINE_STATS = {
    # ── PlantCyc source ────────────────────────────────────────────────────
    'plantcyc_pathways'         : 1_162,
    'plantcyc_reactions'        : 1_316,
    'plantcyc_gpml_total'       : 2_478,
    'ncbi_taxa_total'           : 439,
    'ncbi_taxa_in_gpml'         : 425,
    'ncbi_taxa_only_complex'    : 12,
    'ncbi_taxa_absent'          : 6,     # taxa in GPML absent from NCBITaxon OBO Foundry release
    'ncbi_taxa_affected_nodes'  : 37,    # DataNode instances carrying an absent taxon
    'ncbi_taxa_annotated_total' : 18_762,# total annotated DataNodes (for %-absent calc)
    'plantcyc_orgids'           : 455,
    # ── GPML DataNodes ─────────────────────────────────────────────────────
    'gpml_geneproduct'          : 8_259,
    'gpml_protein'              : 10_715,
    'gpml_metabolite'           : 23_449,
    'gpml_complex_group'        : 1_237,
    'gpml_interactions'         : 34_273,
    'gpml_citations'            : 20_677,
    'geneproduct_annot_pct'     : 98.7,
    'protein_annot_pct'         : 98.8,
    # ── Input validation ───────────────────────────────────────────────────
    'validation_errors'         : 5,     # cross-species products, skipped
    'validation_warnings'       : 2,     # protein-multi-species + ORG-code
    # ── RDF validation (validate_rdf.py) ───────────────────────────────────
    'rdf_validation_errors'     : 0,
    # ── RDF triple counts per named graph ─────────────────────────────────
    'triples_pathways'          : 3_826_567,
    'triples_taxextra'          : 30_176,
    'triples_propextra'         : 2_617_839,
    'triples_bgc_plantismash'   : 11_680,
    'triples_bgc_mibig'         : 1_770,
    'triples_ncbitaxon'         : 17_707,   # MIREOT subset: 424 taxa + ancestors
    'triples_void'              : 124,
    # ── NCBITaxon ROBOT MIREOT subset ─────────────────────────────────────
    'robot_taxa_seed'           : 424,
    'robot_subset_mb'           : 1.4,
    'ncbitaxon_full_mb'         : 1_800,    # full release for comparison
    'ncbitaxon_version'         : '2026-05-13',
    # ── BGC ────────────────────────────────────────────────────────────────
    'bgc_crosslinks'            : 199,
    'bgc_triples_total'         : 13_450,
    # ── Metabolites ────────────────────────────────────────────────────────
    'metabolites_unique'        : 4_577,
    'metabolites_with_inchikey' : 4_111,
}

# Derived totals
PIPELINE_STATS['triples_pathway_graphs'] = (
    PIPELINE_STATS['triples_pathways'] +
    PIPELINE_STATS['triples_taxextra'] +
    PIPELINE_STATS['triples_propextra']
)
PIPELINE_STATS['triples_total'] = (
    PIPELINE_STATS['triples_pathway_graphs'] +
    PIPELINE_STATS['triples_bgc_plantismash'] +
    PIPELINE_STATS['triples_bgc_mibig'] +
    PIPELINE_STATS['triples_ncbitaxon'] +
    PIPELINE_STATS['triples_void']
)

print(f"Total Virtuoso triples:    {PIPELINE_STATS['triples_total']:,}")
print(f"Pathway-relevant triples:  {PIPELINE_STATS['triples_pathway_graphs'] + PIPELINE_STATS['bgc_triples_total']:,}")
print(f"NCBITaxon MIREOT subset:   {PIPELINE_STATS['triples_ncbitaxon']:,} triples  ({PIPELINE_STATS['robot_subset_mb']} MB  ←  {PIPELINE_STATS['ncbitaxon_full_mb']:,} MB full release)")
print(f"Taxa absent from OBO:      {PIPELINE_STATS['ncbi_taxa_absent']} taxa · {PIPELINE_STATS['ncbi_taxa_affected_nodes']} DataNodes affected ({100*PIPELINE_STATS['ncbi_taxa_affected_nodes']/PIPELINE_STATS['ncbi_taxa_annotated_total']:.1f}%)")


Total Virtuoso triples:    6,505,863
Pathway-relevant triples:  6,488,032
NCBITaxon MIREOT subset:   17,707 triples  (1.4 MB  ←  1,800 MB full release)
Taxa absent from OBO:      6 taxa · 37 DataNodes affected (0.2%)


---
## Pipeline transformation table

In [2]:
S = PIPELINE_STATS

rows = [
    {
        'Stage': '1. PlantCyc 17.0 (source)',
        'Tool / Script': 'BioCyc flat files (.dat)',
        'Output artefact': f"{S['plantcyc_orgids']} ORG-IDs → {S['ncbi_taxa_total']} NCBI taxa",
        'Key statistics': f"{S['plantcyc_pathways']:,} pathways · {S['plantcyc_reactions']:,} reactions",
        'Notes': 'PMN/PlantCyc licence',
    },
    {
        'Stage': '2. Input validation',
        'Tool / Script': 'validate_plantcyc_input.py',
        'Output artefact': 'VALIDATION_REPORT.txt, VALIDATION_SUMMARY.tsv',
        'Key statistics': f"{S['validation_errors']} ERRORs skipped · {S['validation_warnings']} WARNINGs",
        'Notes': 'Deterministic build; documented in Table S3',
    },
    {
        'Stage': '3. GPML conversion',
        'Tool / Script': 'build_pathways.py + gpml2rdf-4.0.4.jar',
        'Output artefact': f"{S['plantcyc_gpml_total']:,} GPML2021 files",
        'Key statistics': (f"{S['gpml_geneproduct']:,} GeneProduct · {S['gpml_protein']:,} Protein · "
                           f"{S['gpml_metabolite']:,} Metabolite · {S['gpml_interactions']:,} interactions; "
                           f"Taxon coverage: {S['geneproduct_annot_pct']}% genes · {S['protein_annot_pct']}% proteins"),
        'Notes': 'GPML2021 XSD validated (0 errors)',
    },
    {
        'Stage': '4. Core RDF',
        'Tool / Script': 'gpml-to-rdf (Java + Makefile)',
        'Output artefact': 'graph/pathways',
        'Key statistics': f"{S['triples_pathways']:,} triples",
        'Notes': 'WikiPathways wp: vocabulary',
    },
    {
        'Stage': '5. Taxonomy extra RDF',
        'Tool / Script': 'create_gpml_taxonomy_extra_rdf.py',
        'Output artefact': 'graph/gpml-taxonomy-extra',
        'Key statistics': (f"{S['triples_taxextra']:,} triples · "
                           f"{S['ncbi_taxa_in_gpml']}/{S['ncbi_taxa_total']} NCBI taxa annotated · "
                           f"{S['ncbi_taxa_absent']} taxa absent from OBO Foundry "
                           f"({S['ncbi_taxa_affected_nodes']} DataNodes affected)"),
        'Notes': 'wp:organism ncbi:XXXX per DataNode; absent taxa documented in explore_taxonomy_rdf.ipynb',
    },
    {
        'Stage': '6. Properties extra RDF',
        'Tool / Script': 'create_gpml_properties_extra_rdf.py',
        'Output artefact': 'graph/gpml-properties-extra',
        'Key statistics': f"{S['triples_propextra']:,} triples",
        'Notes': 'PlantCyc key-value metadata preserved',
    },
    {
        'Stage': '7. RDF validation',
        'Tool / Script': 'validate_rdf.py',
        'Output artefact': 'RDF_VALIDATION_REPORT.txt',
        'Key statistics': (f"{S['rdf_validation_errors']} errors · "
                           f"{S['triples_pathways']:,} + {S['triples_taxextra']:,} + "
                           f"{S['triples_propextra']:,} triples across 3 graphs verified"),
        'Notes': 'TTL syntax + content checks on all output files before Virtuoso load',
    },
    {
        'Stage': '8. NCBITaxon ontology (ROBOT MIREOT)',
        'Tool / Script': 'load-ncbitaxon.sh --subset plantmetwiki (Snorql-UI)',
        'Output artefact': 'graph/ncbitaxon',
        'Key statistics': (f"{S['triples_ncbitaxon']:,} triples · "
                           f"MIREOT subset: {S['robot_taxa_seed']} seed taxa + ancestors · "
                           f"{S['robot_subset_mb']} MB "
                           f"(from {S['ncbitaxon_full_mb']:,} MB full release, v{S['ncbitaxon_version']})"),
        'Notes': 'ROBOT v1.9.6 extract --method MIREOT; OBO Foundry CC0 licence',
    },
    {
        'Stage': '9. BGC integration',
        'Tool / Script': 'map-to-rdf',
        'Output artefact': 'graph/bgc-mibig, graph/bgc-plantismash',
        'Key statistics': f"{S['bgc_triples_total']:,} triples · {S['bgc_crosslinks']} pathway crosslinks",
        'Notes': 'MIBiG 4.0 + plantiSMASH predictions',
    },
    {
        'Stage': '10. Triplestore (Virtuoso 7.2)',
        'Tool / Script': 'docker compose up -d virtuoso',
        'Output artefact': 'SPARQL endpoint + Snorql-UI browser',
        'Key statistics': f"{S['triples_total']:,} total triples across 6 named graphs",
        'Notes': 'https://sparql-plantmetwiki.bioinformatics.nl',
    },
]

df = pd.DataFrame(rows)

# Save as TSV
tsv_path = OUT_DIR / 'pipeline_table.tsv'
df.to_csv(tsv_path, sep='\t', index=False)
print(f"Saved: {tsv_path}")

# Display styled HTML in notebook
from IPython.display import display, HTML
styled = (df.style
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap',
                       'font-size': '12px', 'padding': '4px 8px'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color','#2c3e50'),
                                      ('color','white'),('font-size','12px'),
                                      ('padding','6px 10px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f8f9fa')]},
    ])
    .hide(axis='index')
)
display(styled)


Saved: figures/output/pipeline_table.tsv


Stage,Tool / Script,Output artefact,Key statistics,Notes
1. PlantCyc 17.0 (source),BioCyc flat files (.dat),455 ORG-IDs → 439 NCBI taxa,"1,162 pathways · 1,316 reactions",PMN/PlantCyc licence
2. Input validation,validate_plantcyc_input.py,"VALIDATION_REPORT.txt, VALIDATION_SUMMARY.tsv",5 ERRORs skipped · 2 WARNINGs,Deterministic build; documented in Table S3
3. GPML conversion,build_pathways.py + gpml2rdf-4.0.4.jar,"2,478 GPML2021 files","8,259 GeneProduct · 10,715 Protein · 23,449 Metabolite · 34,273 interactions; Taxon coverage: 98.7% genes · 98.8% proteins",GPML2021 XSD validated (0 errors)
4. Core RDF,gpml-to-rdf (Java + Makefile),graph/pathways,"3,826,567 triples",WikiPathways wp: vocabulary
5. Taxonomy extra RDF,create_gpml_taxonomy_extra_rdf.py,graph/gpml-taxonomy-extra,"30,176 triples · 425/439 NCBI taxa annotated · 6 taxa absent from OBO Foundry (37 DataNodes affected)",wp:organism ncbi:XXXX per DataNode; absent taxa documented in explore_taxonomy_rdf.ipynb
6. Properties extra RDF,create_gpml_properties_extra_rdf.py,graph/gpml-properties-extra,"2,617,839 triples",PlantCyc key-value metadata preserved
7. RDF validation,validate_rdf.py,RDF_VALIDATION_REPORT.txt,"0 errors · 3,826,567 + 30,176 + 2,617,839 triples across 3 graphs verified",TTL syntax + content checks on all output files before Virtuoso load
8. NCBITaxon ontology (ROBOT MIREOT),load-ncbitaxon.sh --subset plantmetwiki (Snorql-UI),graph/ncbitaxon,"17,707 triples · MIREOT subset: 424 seed taxa + ancestors · 1.4 MB (from 1,800 MB full release, v2026-05-13)",ROBOT v1.9.6 extract --method MIREOT; OBO Foundry CC0 licence
9. BGC integration,map-to-rdf,"graph/bgc-mibig, graph/bgc-plantismash","13,450 triples · 199 pathway crosslinks",MIBiG 4.0 + plantiSMASH predictions
10. Triplestore (Virtuoso 7.2),docker compose up -d virtuoso,SPARQL endpoint + Snorql-UI browser,"6,505,863 total triples across 6 named graphs",https://sparql-plantmetwiki.bioinformatics.nl


---
## Pipeline figure (Graphviz SVG)

In [3]:
S = PIPELINE_STATS

dot = graphviz.Digraph(
    name='PlantMetWiki_pipeline',
    comment='PlantMetWiki data transformation pipeline',
    format='svg',
    engine='dot',
)
dot.attr(rankdir='TB', nodesep='0.5', ranksep='0.7',
         fontname='Arial', fontsize='11', bgcolor='white')
dot.attr('node', fontname='Arial', fontsize='10', margin='0.15,0.10')
dot.attr('edge', fontname='Arial', fontsize='9', color='#555555')

# ── Colours ───────────────────────────────────────────────────────────────────
C_SRC   = '#dbe9f4'
C_TOOL  = '#fef3cd'
C_GRAPH = '#d5f5e3'
C_ACC   = '#e8daef'

# ── Cluster: source data ─────────────────────────────────────────────────────
with dot.subgraph(name='cluster_source') as c:
    c.attr(label='Source data', style='filled,rounded', fillcolor=C_SRC,
           color='#2980b9', penwidth='1.5', fontcolor='#2980b9', fontsize='11',
           fontname='Arial')
    c.node('plantcyc', shape='cylinder', style='filled', fillcolor='white',
           label=(f'PlantCyc 17.0\n'
                  f'{S["plantcyc_pathways"]:,} pathways · {S["plantcyc_reactions"]:,} reactions\n'
                  f'{S["plantcyc_orgids"]} ORG-IDs → {S["ncbi_taxa_total"]} NCBI taxa'))
    c.node('mibig', shape='cylinder', style='filled', fillcolor='white',
           label='MIBiG 4.0\n44 plant BGCs')
    c.node('plantismash', shape='cylinder', style='filled', fillcolor='white',
           label='plantiSMASH\n128 predicted BGCs')
    c.node('ncbitaxon_src', shape='cylinder', style='filled', fillcolor='white',
           label='NCBITaxon (OBO Foundry)\nCC0 licence')
    c.node('wikidata_src', shape='cylinder', style='filled', fillcolor='white',
           label='Wikidata\n(federated via InChIKey)')

# ── Cluster: processing tools ─────────────────────────────────────────────────
with dot.subgraph(name='cluster_tools') as c:
    c.attr(label='Pipeline tools', style='filled,rounded', fillcolor=C_TOOL,
           color='#e67e22', penwidth='1.5', fontcolor='#e67e22', fontsize='11',
           fontname='Arial')
    c.node('validate', shape='diamond', style='filled', fillcolor='white',
           label=(f'validate_plantcyc_input.py\n'
                  f'{S["validation_errors"]} ERRORs (skipped) · {S["validation_warnings"]} WARNINGs'))
    c.node('cyc2wiki', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'Cyc_to_wiki + build_pathways.py\n'
                  f'{S["plantcyc_gpml_total"]:,} GPML2021 files\n'
                  f'{S["gpml_geneproduct"]:,} GeneProduct · {S["gpml_protein"]:,} Protein\n'
                  f'{S["gpml_metabolite"]:,} Metabolite · {S["gpml_interactions"]:,} interactions'))
    c.node('xsd_val', shape='box', style='filled,rounded', fillcolor='white',
           label='test_gpml_files.py\nGPML2021 XSD validation\n0 errors')
    c.node('gpml2rdf', shape='box', style='filled,rounded', fillcolor='white',
           label=('gpml-to-rdf  (Java + Python scripts)\n'
                  'gpml2rdf-4.0.4.jar · taxonomy extra · properties extra\n'
                  f'Core {S["triples_pathways"]:,} + Tax {S["triples_taxextra"]:,} + Prop {S["triples_propextra"]:,} triples'))
    c.node('rdf_validate', shape='diamond', style='filled', fillcolor='white',
           label=(f'validate_rdf.py\n'
                  f'{S["rdf_validation_errors"]} errors · {S["triples_pathway_graphs"]:,} triples verified'))
    c.node('load_ncbi', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'load-ncbitaxon.sh --subset plantmetwiki\n'
                  f'ROBOT v1.9.6  extract --method MIREOT\n'
                  f'{S["robot_taxa_seed"]} seed taxa + ancestors  →  {S["triples_ncbitaxon"]:,} triples\n'
                  f'{S["robot_subset_mb"]} MB  (full release: {S["ncbitaxon_full_mb"]:,} MB)'))
    c.node('map2rdf', shape='box', style='filled,rounded', fillcolor='white',
           label=(f'map-to-rdf\n'
                  f'{S["bgc_crosslinks"]} BGC-pathway crosslinks · {S["bgc_triples_total"]:,} triples'))

# ── Cluster: Virtuoso named graphs ────────────────────────────────────────────
with dot.subgraph(name='cluster_virtuoso') as c:
    c.attr(label=f'Virtuoso 7.2  —  {S["triples_total"]:,} total triples  (6 named graphs)',
           style='filled,rounded', fillcolor=C_GRAPH,
           color='#27ae60', penwidth='1.5', fontcolor='#27ae60', fontsize='11',
           fontname='Arial')
    c.node('g_pathways', shape='tab', style='filled', fillcolor='white',
           label=f'graph/pathways\n{S["triples_pathways"]:,} triples')
    c.node('g_taxextra', shape='tab', style='filled', fillcolor='white',
           label=(f'graph/gpml-taxonomy-extra\n{S["triples_taxextra"]:,} triples\n'
                  f'{S["ncbi_taxa_in_gpml"]}/{S["ncbi_taxa_total"]} taxa · '
                  f'{S["ncbi_taxa_absent"]} absent from OBO Foundry'))
    c.node('g_propextra', shape='tab', style='filled', fillcolor='white',
           label=f'graph/gpml-properties-extra\n{S["triples_propextra"]:,} triples')
    c.node('g_bgc', shape='tab', style='filled', fillcolor='white',
           label=f'graph/bgc-*\n{S["bgc_triples_total"]:,} triples')
    c.node('g_ncbi', shape='tab', style='filled', fillcolor='white',
           label=(f'graph/ncbitaxon\n{S["triples_ncbitaxon"]:,} triples\n'
                  f'MIREOT subset  v{S["ncbitaxon_version"]}'))
    c.node('sparql', shape='component', style='filled', fillcolor='white',
           label='SPARQL endpoint\nhttps://sparql-plantmetwiki.bioinformatics.nl')

# ── Cluster: access & results (right column, beside tools+virtuoso) ───────────
with dot.subgraph(name='cluster_access') as c:
    c.attr(label='Access & results', style='filled,rounded', fillcolor=C_ACC,
           color='#8e44ad', penwidth='1.5', fontcolor='#8e44ad', fontsize='11',
           fontname='Arial')
    c.node('snorql', shape='box', style='filled,rounded', fillcolor='white',
           label='Snorql-UI\nhttps://plantmetwiki.bioinformatics.nl')
    c.node('sparql_examples', shape='note', style='filled', fillcolor='white',
           label='SPARQL query examples\ngithub.com/pathway-lod/SPARQLQueries\ncustomizable via Snorql-UI settings')
    c.node('tutorials', shape='note', style='filled', fillcolor='white',
           label='Tutorial pages\nStep-by-step SPARQL\nfederated queries')
    c.node('notebooks', shape='note', style='filled', fillcolor='white',
           label=(f'Jupyter notebooks\n'
                  f'{S["metabolites_unique"]:,} metabolites ({S["metabolites_with_inchikey"]:,} with InChIKey)\n'
                  f'Figures · Cross-species inference · Federated queries'))

# ── Rank constraints: access cluster beside pipeline (not below) ──────────────
with dot.subgraph() as s:
    s.attr(rank='same')
    s.node('validate')
    s.node('snorql')
with dot.subgraph() as s:
    s.attr(rank='same')
    s.node('sparql')
    s.node('notebooks')

# ── Edges: pipeline flow ──────────────────────────────────────────────────────
dot.edge('plantcyc', 'validate')
dot.edge('validate', 'cyc2wiki', label='5 ERRORs skipped')
dot.edge('cyc2wiki', 'xsd_val', label='2,478 GPML files')
dot.edge('xsd_val', 'gpml2rdf', label='XSD pass')
dot.edge('gpml2rdf', 'rdf_validate', label='TTL output')
dot.edge('rdf_validate', 'g_pathways', label='0 errors')
dot.edge('rdf_validate', 'g_taxextra')
dot.edge('rdf_validate', 'g_propextra')
dot.edge('mibig', 'map2rdf')
dot.edge('plantismash', 'map2rdf')
dot.edge('map2rdf', 'g_bgc')
dot.edge('ncbitaxon_src', 'load_ncbi')
dot.edge('load_ncbi', 'g_ncbi')
dot.edge('g_pathways', 'sparql')
dot.edge('g_taxextra', 'sparql')
dot.edge('g_propextra', 'sparql')
dot.edge('g_bgc', 'sparql')
dot.edge('g_ncbi', 'sparql')
# Access edges — constraint=false so these don't push the access cluster below Virtuoso
dot.edge('sparql', 'snorql', constraint='false')
dot.edge('sparql', 'notebooks', constraint='false')
dot.edge('snorql', 'sparql_examples', style='dashed', constraint='false', label='loads from GitHub')
dot.edge('snorql', 'tutorials', style='dashed', constraint='false')
# Single federated SPARQL edge
dot.edge('wikidata_src', 'notebooks', style='dashed', label='federated SPARQL', constraint='false')

# ── Render ────────────────────────────────────────────────────────────────────
dot_src_path = str(FIG_DIR / 'pipeline_overview')
dot.render(dot_src_path, cleanup=False)
(FIG_DIR / 'pipeline_overview.dot').write_text(dot.source, encoding='utf-8')

print(f"Saved SVG: {dot_src_path}.svg")
print(f"Saved DOT: {dot_src_path}.dot")


Saved SVG: figures/output/figures/pipeline_overview.svg
Saved DOT: figures/output/figures/pipeline_overview.dot


---
## Figure caption / interpretation

**Figure legend — PlantMetWiki data transformation pipeline (10 stages).**

The diagram shows the four regions of the PlantMetWiki knowledge-graph construction pipeline:

- **Source data** (blue, stage 1): PlantCyc 17.0 biochemical databases (1,162 pathways, 1,316 reactions, 455 ORG-IDs covering 439 NCBI taxa), MIBiG 4.0 and plantiSMASH BGC annotations, the OBO Foundry NCBITaxon ontology (CC0), and Wikidata (accessed via federated SPARQL using InChIKey).

- **Pipeline tools** (yellow, stages 2–9): Input validation (stage 2) checks taxonomic consistency before file generation. GPML conversion (stage 3) produces 2,478 GPML2021 files. GPML-to-RDF (stages 4–6) generates three named graphs totalling 6,474,582 triples: core WikiPathways RDF (3,826,567), taxonomy extra (30,176; 424/439 NCBI taxa annotated; 6 taxa absent from OBO Foundry release, affecting 37 DataNodes), and properties extra (2,617,839). RDF validation (stage 7) checks all output TTL files for syntax and content errors (0 errors). The NCBITaxon ontology step (stage 8) uses ROBOT v1.9.6 MIREOT extraction to build a data-driven subset of 424 seed taxa plus full ancestor lineages (17,707 triples, 1.4 MB — reduced from the 1.8 GB full release). BGC integration (stage 9) adds 199 biosynthetic gene cluster crosslinks.

- **Virtuoso triplestore** (green, stage 10): Six named graphs totalling 6,505,863 triples. The graph/ncbitaxon named graph provides taxon labels and hierarchical context for all PlantMetWiki taxa.

- **Access and results** (purple): A public SPARQL endpoint exposes all graphs. Snorql-UI provides a browser query interface and loads a curated set of customizable SPARQL query examples from GitHub (github.com/pathway-lod/SPARQLQueries). Tutorial pages document step-by-step SPARQL queries including federated queries linking PlantMetWiki taxa to Wikidata metabolites. Jupyter notebooks support publication figures, cross-species pathway inference, and federated metabolite analyses (4,577 unique metabolites, 4,111 with InChIKey).

The DOT source file (`pipeline_overview.dot`) is version-controlled alongside this notebook.
